In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [2]:
from google.colab import files

In [3]:
# Load all datasets
import zipfile
import os

# Define the path to your zip file
zip_file_path = '/content/northstar_dataset (1).zip'
# Define a new extraction base directory
extraction_base_path = '/content/extracted_northstar_data/'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_base_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_base_path)

print(f"Dataset extracted to: {extraction_base_path}")

# Assuming the zip file contains a top-level folder named 'northstar_dataset'
# This will be the actual path where the CSV files reside
data_folder_name = 'northstar_dataset'
data_folder_path = os.path.join(extraction_base_path, data_folder_name)

# Now, load the CSV files from the correct path
customers = pd.read_csv(os.path.join(data_folder_path, 'customers.csv'))
orders = pd.read_csv(os.path.join(data_folder_path, 'orders.csv'))
deliveries = pd.read_csv(os.path.join(data_folder_path, 'deliveries.csv'))
complaints = pd.read_csv(os.path.join(data_folder_path, 'complaints.csv'))
drivers = pd.read_csv(os.path.join(data_folder_path, 'drivers.csv'))
vehicles = pd.read_csv(os.path.join(data_folder_path, 'vehicles.csv'))
hubs = pd.read_csv(os.path.join(data_folder_path, 'hubs.csv'))
incidents = pd.read_csv(os.path.join(data_folder_path, 'incidents.csv'))
app_events = pd.read_csv(os.path.join(data_folder_path, 'app_events.csv'))

print("All datasets loaded successfully!")
print(f"Customers: {customers.shape}")
print(f"Orders: {orders.shape}")
print(f"Deliveries: {deliveries.shape}")

Dataset extracted to: /content/extracted_northstar_data/
All datasets loaded successfully!
Customers: (650, 9)
Orders: (1250, 11)
Deliveries: (950, 13)


In [4]:
#Data Cleaning - Standardise categorical values
def standardise_zone(zone):
    if pd.isna(zone):
        return np.nan
    zone = str(zone).upper().strip()
    zone_map = {
        'NORTH': 'NORTH', 'NTH': 'NORTH', 'NORTHERN': 'NORTH',
        'SOUTH': 'SOUTH', 'STH': 'SOUTH', 'SOUTHERN': 'SOUTH',
        'EAST': 'EAST', 'EST': 'EAST', 'EASTERN': 'EAST',
        'WEST': 'WEST', 'WST': 'WEST', 'WES': 'WEST', 'WESTERN': 'WEST',
        'CENTRAL': 'CENTRAL', 'CTR': 'CENTRAL', 'CENTER': 'CENTRAL',
        'RIVERSIDE': 'RIVERSIDE', 'RIVER': 'RIVERSIDE', 'RVSIDE': 'RIVERSIDE',
        'AIRPORT': 'AIRPORT', 'AIRPRT': 'AIRPORT', 'APT': 'AIRPORT'
    }
    return zone_map.get(zone, zone)

In [5]:
# Apply to relevant columns
if 'zone_context' in app_events.columns:
    app_events['zone_context_std'] = app_events['zone_context'].apply(standardise_zone)

if 'home_zone' in customers.columns:
    customers['home_zone_std'] = customers['home_zone'].apply(standardise_zone)

print("Zone names standardised!")

Zone names standardised!


In [6]:
# Handle missing values
print("\n=== MISSING VALUES BEFORE CLEANING ===")
print(customers.isnull().sum())


=== MISSING VALUES BEFORE CLEANING ===
customer_id              0
age                      0
home_zone                0
customer_type            0
signup_date              0
loyalty_score           20
app_engagement_score     0
preferred_channel       13
account_status           0
home_zone_std            0
dtype: int64


In [7]:
# Fill missing values
customers['preferred_channel'].fillna('Unknown', inplace=True)
customers['loyalty_score'].fillna(customers['loyalty_score'].median(), inplace=True)
customers['app_engagement_score'].fillna(customers['app_engagement_score'].median(), inplace=True)

drivers['training_score'].fillna(drivers['training_score'].median(), inplace=True)
vehicles['battery_health_pct'].fillna(vehicles['battery_health_pct'].median(), inplace=True)

print("\n=== MISSING VALUES AFTER CLEANING ===")
print(customers.isnull().sum())


=== MISSING VALUES AFTER CLEANING ===
customer_id             0
age                     0
home_zone               0
customer_type           0
signup_date             0
loyalty_score           0
app_engagement_score    0
preferred_channel       0
account_status          0
home_zone_std           0
dtype: int64


/tmp/ipykernel_8111/3362880650.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customers['preferred_channel'].fillna('Unknown', inplace=True)
/tmp/ipykernel_8111/3362880650.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

In [8]:
# Convert date columns
deliveries['dispatch_time'] = pd.to_datetime(deliveries['dispatch_time'])
deliveries['delivery_completed_at'] = pd.to_datetime(deliveries['delivery_completed_at'])

In [9]:
# Calculate delivery duration in hours
deliveries['delivery_duration_hours'] = (deliveries['delivery_completed_at'] - deliveries['dispatch_time']).dt.total_seconds() / 3600

In [10]:
# critical failure flag
deliveries['is_critical_failure'] = ((deliveries['delivery_status'] == 'Failed') &
                                       (deliveries['customer_rating_post_delivery'] < 2.0)).astype(int)

In [11]:
# Aggregate incidents per delivery
incident_summary = incidents.groupby('delivery_id').agg({
    'incident_id': 'count',
    'severity': lambda x: x.mode()[0] if len(x) > 0 else None
}).rename(columns={'incident_id': 'incident_count', 'severity': 'max_severity'})

In [12]:
# Merge incident summary with deliveries
deliveries = deliveries.merge(incident_summary, on='delivery_id', how='left')
deliveries['incident_count'].fillna(0, inplace=True)

print("\n=== FEATURE ENGINEERING COMPLETE ===")
print(f"New features added: delivery_duration_hours, is_critical_failure, incident_count")
print(f"Deliveries with critical failures: {deliveries['is_critical_failure'].sum()}")


=== FEATURE ENGINEERING COMPLETE ===
New features added: delivery_duration_hours, is_critical_failure, incident_count
Deliveries with critical failures: 14


/tmp/ipykernel_8111/2420984840.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  deliveries['incident_count'].fillna(0, inplace=True)


In [13]:
# summary statistics for report
print("\n=== SUMMARY STATISTICS ===")
print(f"Total Customers: {len(customers)}")
print(f"Active Customers: {len(customers[customers['account_status'] == 'Active'])}")
print(f"Total Orders: {len(orders)}")
print(f"Total Deliveries: {len(deliveries)}")
print(f"Delivery Success Rate: {100 * len(deliveries[deliveries['delivery_status'] == 'OnTime']) / len(deliveries):.1f}%")
print(f"Average Customer Rating: {deliveries['customer_rating_post_delivery'].mean():.2f}/5.0")


=== SUMMARY STATISTICS ===
Total Customers: 650
Active Customers: 552
Total Orders: 1250
Total Deliveries: 950
Delivery Success Rate: 64.8%
Average Customer Rating: 3.86/5.0


In [14]:
# Transform app_events into nested document structure
app_events_nested = app_events.groupby('session_id').agg({
    'event_id': list,
    'event_type': list,
    'event_timestamp': list,
    'api_latency_ms': list,
    'success_flag': list,
    'customer_id': 'first',
    'device_type': 'first',
    'zone_context': 'first'
}).reset_index()

In [15]:
# Rename columns for nested structure
app_events_nested.rename(columns={
    'event_id': 'event_ids',
    'event_type': 'event_types',
    'event_timestamp': 'event_timestamps',
    'api_latency_ms': 'api_latencies',
    'success_flag': 'success_flags'
}, inplace=True)

print(f"\n=== MONGODB PREPARATION ===")
print(f"Original app_events: {len(app_events)} rows")
print(f"Nested app_sessions: {len(app_events_nested)} documents")
print(f"Sample nested document structure:")
print(app_events_nested.head(1).to_dict('records')[0])


=== MONGODB PREPARATION ===
Original app_events: 640 rows
Nested app_sessions: 637 documents
Sample nested document structure:
{'session_id': 'S10192', 'event_ids': ['AE00403'], 'event_types': ['payment_retry'], 'event_timestamps': ['2025-03-19 09:20:00'], 'api_latencies': [108], 'success_flags': [1], 'customer_id': 'C0180', 'device_type': 'Android', 'zone_context': 'Ctr'}


In [16]:
# Save cleaned data for MongoDB import
app_events_nested.to_json('app_sessions_nested.json', orient='records', indent=2)
deliveries_cleaned = deliveries[['delivery_id', 'order_id', 'delivery_status',
                                  'customer_rating_post_delivery', 'fuel_or_charge_cost',
                                  'is_critical_failure', 'incident_count']]
deliveries_cleaned.to_csv('deliveries_cleaned.csv', index=False)

print("\nCleaned data saved for MongoDB import!")
print("Files created: app_sessions_nested.json, deliveries_cleaned.csv")


Cleaned data saved for MongoDB import!
Files created: app_sessions_nested.json, deliveries_cleaned.csv
